In [ ]:
from pathlib import Path
from collections import Counter

ROOT = Path.cwd()
CLAIMS_DIR = ROOT / "841403b5-bb4b-4e2a-a73f-570c1b1af8fb" / "Claims"
OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Workspace root:", ROOT)
print("Claims dir exists:", CLAIMS_DIR.exists())
print("Outputs dir:", OUT_DIR)

In [ ]:
# couting files to understand data composition 
all_files = [p for p in CLAIMS_DIR.rglob('*') if p.is_file()]
ext_counts = Counter(p.suffix.lower() for p in all_files)

print("Total files:", len(all_files))
print("Top extensions:")
for ext, cnt in ext_counts.most_common(15):
    print(f"  {ext or '[no extension]'}: {cnt}")

In [ ]:
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

pdf_files = sorted(CLAIMS_DIR.rglob('*.pdf'))
img_files = sorted([
    p for p in CLAIMS_DIR.rglob('*')
    if p.is_file() and p.suffix.lower() in img_exts
])

sample_pdf = pdf_files[0] if pdf_files else None
sample_img = img_files[0] if img_files else None

print("PDF files found:", len(pdf_files))
print("Image files found:", len(img_files))
print("Sample PDF:", sample_pdf)
print("Sample Image:", sample_img)

In [ ]:
import numpy as np

def _get_easyocr_reader(languages=None, gpu=False):
    import easyocr
    langs = languages or ["en"]
    return easyocr.Reader(langs, gpu=gpu)

def _ocr_image_with_easyocr(pil_image, reader):
    arr = np.array(pil_image)
    results = reader.readtext(arr, detail=0, paragraph=True)
    return "\n".join([r for r in results if isinstance(r, str)]).strip()

def extract_text_from_pdf(pdf_path, max_pages=2, use_gpu=False):
    text = ""
    used_method = "none"

    # Stage 1: direct text extraction
    try:
        from pypdf import PdfReader
        reader = PdfReader(str(pdf_path))
        chunks = []
        for page in reader.pages[:max_pages]:
            chunks.append((page.extract_text() or "").strip())
        text = "\n".join(chunks).strip()
        used_method = "pypdf"
    except Exception:
        pass

    # Stage 2: OCR fallback if direct text is too short
    if len(text) < 120:
        try:
            from pdf2image import convert_from_path

            images = convert_from_path(str(pdf_path), first_page=1, last_page=max_pages)
            ocr_reader = _get_easyocr_reader(languages=["en"], gpu=use_gpu)
            ocr_chunks = [_ocr_image_with_easyocr(img, ocr_reader) for img in images]
            ocr_text = "\n".join([c for c in ocr_chunks if c]).strip()

            if len(ocr_text) > len(text):
                text = ocr_text
                used_method = "pdf2image+easyocr"
        except Exception:
            pass

    return text, used_method

In [ ]:
import os
import numpy as np
from pathlib import Path

def _get_local_easyocr_reader(project_root, languages=None, gpu=False):
    import easyocr

    langs = languages or ["en"]
    model_dir = Path(project_root) / "assets" / "easyocr_models"
    user_net_dir = Path(project_root) / "assets" / "easyocr_user_network"
    model_dir.mkdir(parents=True, exist_ok=True)
    user_net_dir.mkdir(parents=True, exist_ok=True)

    os.environ["EASYOCR_MODULE_PATH"] = str(model_dir)
    os.environ["MODULE_PATH"] = str(model_dir)

    return easyocr.Reader(
        langs,
        gpu=gpu,
        model_storage_directory=str(model_dir),
        user_network_directory=str(user_net_dir),
        download_enabled=False,
    )

def _ocr_image_with_easyocr(pil_image, reader):
    arr = np.array(pil_image)
    results = reader.readtext(arr, detail=0, paragraph=True)
    return "\n".join([r for r in results if isinstance(r, str)]).strip()

def extract_text_from_pdf(pdf_path, max_pages=2, use_gpu=False, debug=False):
    """Extract text from PDF with fallback OCR."""
    text = ""
    used_method = "none"

    # Stage 1: direct text extraction via pypdf
    try:
        from pypdf import PdfReader
        reader = PdfReader(str(pdf_path))
        chunks = []
        for page in reader.pages[:max_pages]:
            chunks.append((page.extract_text() or "").strip())
        text = "\n".join(chunks).strip()
        used_method = "pypdf"
        if debug:
            print(f"  Direct extraction: {len(text)} chars")
    except Exception as e:
        if debug:
            print(f"  Direct text extraction failed: {repr(e)}")

    # Stage 2: OCR fallback if direct text is too short
    if len(text) < 120:
        try:
            from pdf2image import convert_from_path
            
            if debug:
                print(f"  Converting PDF to images...")
            
            images = convert_from_path(str(pdf_path), first_page=1, last_page=max_pages)
            
            if debug:
                print(f"  Pages converted: {len(images)}")

            ocr_reader = _get_local_easyocr_reader(ROOT, languages=["en"], gpu=use_gpu)
            ocr_chunks = [_ocr_image_with_easyocr(img, ocr_reader) for img in images]
            ocr_text = "\n".join([c for c in ocr_chunks if c]).strip()

            if debug:
                print(f"  OCR extraction: {len(ocr_text)} chars")

            if len(ocr_text) > len(text):
                text = ocr_text
                used_method = "pdf2image+easyocr(local-only)"
        except Exception as e:
            if debug:
                print(f"  OCR fallback failed: {repr(e)}")

    return text, used_method

In [ ]:
# Cell 5A: Offline preflight (no installs, no downloads)
from pathlib import Path
import importlib
import shutil

print("=== Offline Preflight ===")
required = ["pypdf", "pdf2image", "easyocr", "PIL", "cv2", "numpy", "torch"]
missing = []

for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"OK: {pkg}")
    except Exception as e:
        print(f"MISSING: {pkg} ({e})")
        missing.append(pkg)

print("\n=== Local Assets Check ===")
model_dir = ROOT / "assets" / "easyocr_models"
user_net_dir = ROOT / "assets" / "easyocr_user_network"
poppler_on_path = shutil.which("pdfinfo")

print("easyocr model dir:", model_dir, "exists:", model_dir.exists())
print("easyocr user network dir:", user_net_dir, "exists:", user_net_dir.exists())
print("pdfinfo on PATH:", poppler_on_path if poppler_on_path else "NOT FOUND")

if missing:
    print("\nRESULT: FAIL (missing python packages)")
elif not poppler_on_path:
    
    print("\nRESULT: FAIL (Poppler not on PATH)")
else:
    print("\nRESULT: PASS (offline prerequisites available)")

print("\nNote: This notebook does NOT install anything automatically.")

In [ ]:
# Cell 5B: Local-only OCR readiness diagnostics
import os
from pathlib import Path

model_dir = ROOT / "assets" / "easyocr_models"
user_net_dir = ROOT / "assets" / "easyocr_user_network"
model_dir.mkdir(parents=True, exist_ok=True)
user_net_dir.mkdir(parents=True, exist_ok=True)

os.environ["EASYOCR_MODULE_PATH"] = str(model_dir)
os.environ["MODULE_PATH"] = str(model_dir)

print("EASYOCR_MODULE_PATH:", os.environ.get("EASYOCR_MODULE_PATH"))
print("MODULE_PATH:", os.environ.get("MODULE_PATH"))
print("Model directory:", model_dir)
print("User network directory:", user_net_dir)
print("Directory file count (models):", len(list(model_dir.glob("*"))))

if len(list(model_dir.glob("*"))) == 0:
    print("WARNING: easyocr model files are not present locally yet.")
    print("For strict offline evaluation, place EasyOCR weights in assets/easyocr_models before running OCR.")
else:
    print("OK: local model files detected.")

In [ ]:
# # Cell 5C: Run OCR on one sample PDF and save output (offline-safe)
# if sample_pdf is None:
#     print("No PDF found. Check dataset path or extension cases.")
# else:
#     text, method = extract_text_from_pdf(sample_pdf, max_pages=2, use_gpu=False)
#     out_txt = OUT_DIR / "sample_ocr_text_easyocr.txt"
#     out_txt.write_text(text, encoding="utf-8", errors="ignore")

#     print("OCR method used:", method)
#     print("Characters extracted:", len(text))
#     print("Saved to:", out_txt)
#     print("--- OCR preview (first 1200 chars) ---")
#     print(text[:1200] if text else "[No text extracted]")

#     if len(text) == 0:
#         print("\nALERT: No text extracted.")
#         print("Possible reasons:")
#         print("  1. PDF has no extractable text and OCR fallback prerequisites are missing")
#         print("  2. Poppler (pdfinfo) is not available")
#         print("  3. EasyOCR local model files are not present in assets/easyocr_models")
#         print("\nRun Cell 5A and Cell 5B for clear offline diagnostics.")

# 24-not working 

In [ ]:
# this was not working since some ocr dependecies were on some other libs ...

# import csv
# import hashlib
# from pathlib import Path

# MAX_PDFS = 5
# LOW_QUALITY_THRESHOLD = 200

# # Self-healing setup so this cell can run independently.
# if "ROOT" not in globals():
#     ROOT = Path.cwd()
# if "CLAIMS_DIR" not in globals():
#     CLAIMS_DIR = ROOT / "dataset" / "Claims"
# if "OUT_DIR" not in globals():
#     OUT_DIR = ROOT / "outputs"
#     OUT_DIR.mkdir(parents=True, exist_ok=True)
# if "pdf_files" not in globals() or not isinstance(pdf_files, list):
#     pdf_files = sorted(CLAIMS_DIR.rglob("*.pdf"))

# if "extract_text_from_pdf" not in globals():
#     raise RuntimeError("extract_text_from_pdf is not defined. Run Cell 4 code first, then run this cell.")

# ocr_text_dir = OUT_DIR / "ocr_texts"
# ocr_text_dir.mkdir(parents=True, exist_ok=True)

# selected_pdfs = pdf_files[:MAX_PDFS]
# records = []

# print("Tip: If you edited OCR utility recently, re-run Cell 4 before this batch cell.")
# print(f"Workspace root: {ROOT}")
# print(f"Claims dir: {CLAIMS_DIR}")
# print(f"Total PDF files available: {len(pdf_files)}")
# print(f"Processing first {len(selected_pdfs)} PDF files...")

# for idx, pdf_path in enumerate(selected_pdfs, start=1):
#     try:
#         # Backward-compatible call: works with both old/new function signatures.
#         try:
#             text, method = extract_text_from_pdf(pdf_path, max_pages=2, use_gpu=False, debug=False)
#         except TypeError:
#             text, method = extract_text_from_pdf(pdf_path, max_pages=2, use_gpu=False)

#         char_count = len(text)
#         low_quality = char_count < LOW_QUALITY_THRESHOLD

#         # Stable unique filename to avoid collisions across folders.
#         short_hash = hashlib.md5(str(pdf_path).encode("utf-8")).hexdigest()[:10]
#         txt_name = f"{pdf_path.stem}_{short_hash}.txt"
#         txt_path = ocr_text_dir / txt_name
#         txt_path.write_text(text, encoding="utf-8", errors="ignore")

#         records.append({
#             "file_path": str(pdf_path),
#             "ocr_method": method,
#             "char_count": char_count,
#             "low_quality": low_quality,
#             "text_file": str(txt_path),
#         })

#         print(f"[{idx}/{len(selected_pdfs)}] OK | chars={char_count:4d} | method={method} | {pdf_path.name}")
#     except Exception as e:
#         records.append({
#             "file_path": str(pdf_path),
#             "ocr_method": "error",
#             "char_count": 0,
#             "low_quality": True,
#             "text_file": "",
#             "error": repr(e),
#         })
#         print(f"[{idx}/{len(selected_pdfs)}] ERROR | {pdf_path.name} | {repr(e)}")

# csv_path = OUT_DIR / "ocr_batch_results.csv"
# fieldnames = ["file_path", "ocr_method", "char_count", "low_quality", "text_file", "error"]
# with open(csv_path, "w", newline="", encoding="utf-8") as f:
#     writer = csv.DictWriter(f, fieldnames=fieldnames)
#     writer.writeheader()
#     for row in records:
#         writer.writerow({k: row.get(k, "") for k in fieldnames})

# num_low = sum(1 for r in records if r.get("low_quality", True))
# num_error = sum(1 for r in records if r.get("ocr_method") == "error")

# print("\n=== Batch OCR Summary ===")
# print("Processed:", len(records))
# print("Low-quality rows:", num_low)
# print("Error rows:", num_error)
# print("CSV saved:", csv_path)
# print("Text outputs dir:", ocr_text_dir)

In [ ]:
# Fresh all-in-one OCR batch cell (run as-is)

import os
import re
import csv
import sys
import hashlib
import subprocess
from pathlib import Path

# -----------------------------
# 0) Install missing packages
# -----------------------------
def ensure_pkg(pkg_name, import_name=None):
    import_name = import_name or pkg_name
    try:
        __import__(import_name)
        return True
    except Exception:
        print(f"Installing {pkg_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg_name])
        __import__(import_name)
        return True

ensure_pkg("pypdf")
ensure_pkg("pdf2image")
ensure_pkg("numpy")
ensure_pkg("Pillow", "PIL")

# EasyOCR stack (only if needed for scanned PDFs)
# This env var avoids the torchvision nms registration crash in mismatched environments.
os.environ["TORCHVISION_DISABLE_NMS_OP"] = "1"

easyocr_available = True
try:
    ensure_pkg("easyocr")
except Exception as e:
    easyocr_available = False
    print("EasyOCR install/import failed. OCR fallback may be unavailable.")
    print("Reason:", repr(e))

import numpy as np
from pypdf import PdfReader
from pdf2image import convert_from_path

# -----------------------------
# 1) Resolve paths robustly
# -----------------------------
ROOT = Path.cwd()

def find_claims_dir(start: Path) -> Path:
    # Common candidates first
    candidates = [
        start / "841403b5-bb4b-4e2a-a73f-570c1b1af8fb" / "Claims",
        start / "Claims",
    ]
    for c in candidates:
        if c.exists() and c.is_dir():
            return c

    # Recursive fallback (first directory named Claims)
    for p in start.rglob("Claims"):
        if p.is_dir():
            return p

    raise FileNotFoundError("Could not find Claims directory from current working directory.")

CLAIMS_DIR = find_claims_dir(ROOT)
OUT_DIR = ROOT / "outputs"
OCR_TEXT_DIR = OUT_DIR / "ocr_texts"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OCR_TEXT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 2) OCR utilities
# -----------------------------
_EASYOCR_READER = None

def get_easyocr_reader(project_root: Path, gpu: bool = False):
    global _EASYOCR_READER
    if _EASYOCR_READER is not None:
        return _EASYOCR_READER

    if not easyocr_available:
        return None

    import easyocr

    model_dir = project_root / "assets" / "easyocr_models"
    user_net_dir = project_root / "assets" / "easyocr_user_network"
    model_dir.mkdir(parents=True, exist_ok=True)
    user_net_dir.mkdir(parents=True, exist_ok=True)

    os.environ["EASYOCR_MODULE_PATH"] = str(model_dir)
    os.environ["MODULE_PATH"] = str(model_dir)

    _EASYOCR_READER = easyocr.Reader(
        ["en"],
        gpu=gpu,
        model_storage_directory=str(model_dir),
        user_network_directory=str(user_net_dir),
        download_enabled=False,  # strict offline mode
    )
    return _EASYOCR_READER

def ocr_image_easyocr(pil_img, reader):
    arr = np.array(pil_img)
    lines = reader.readtext(arr, detail=0, paragraph=True)
    return "\n".join([x for x in lines if isinstance(x, str)]).strip()

def extract_text_from_pdf(pdf_path: Path, max_pages: int = 3, use_gpu: bool = False, debug: bool = False):
    """
    Returns: (text, method, error_note)
    method in {"pypdf", "pdf2image+easyocr", "none"}
    """
    text = ""
    method = "none"
    err_note = ""

    # Stage A: embedded text
    try:
        reader = PdfReader(str(pdf_path))
        chunks = []
        pages = reader.pages[:max_pages]
        for pg in pages:
            chunks.append((pg.extract_text() or "").strip())
        direct_text = "\n".join([c for c in chunks if c]).strip()

        if len(direct_text) > 0:
            text = direct_text
            method = "pypdf"
            if debug:
                print(f"[{pdf_path.name}] direct chars={len(text)}")
    except Exception as e:
        err_note = f"pypdf_error={repr(e)}"
        if debug:
            print(f"[{pdf_path.name}] pypdf failed:", repr(e))

    # Stage B: OCR fallback for scanned PDFs
    if len(text) < 80:
        try:
            reader = get_easyocr_reader(ROOT, gpu=use_gpu)
            if reader is None:
                if not err_note:
                    err_note = "easyocr_unavailable"
            else:
                images = convert_from_path(str(pdf_path), first_page=1, last_page=max_pages)
                ocr_chunks = [ocr_image_easyocr(img, reader) for img in images]
                ocr_text = "\n".join([c for c in ocr_chunks if c]).strip()

                if len(ocr_text) > len(text):
                    text = ocr_text
                    method = "pdf2image+easyocr"

                if debug:
                    print(f"[{pdf_path.name}] ocr chars={len(ocr_text)}")
        except Exception as e:
            note = f"ocr_error={repr(e)}"
            err_note = f"{err_note}; {note}" if err_note else note
            if debug:
                print(f"[{pdf_path.name}] OCR failed:", repr(e))

    return text, method, err_note

# -----------------------------
# 3) Batch run on all PDFs
# -----------------------------
pdf_files = sorted(CLAIMS_DIR.rglob("*.pdf"))
if not pdf_files:
    raise RuntimeError(f"No PDF files found under: {CLAIMS_DIR}")

LOW_QUALITY_THRESHOLD = 200
MAX_PAGES_PER_PDF = 3  # increase to 4/5 for better OCR coverage if needed

print("=" * 72)
print("BATCH OCR START")
print(f"Root: {ROOT}")
print(f"Claims dir: {CLAIMS_DIR}")
print(f"Total PDFs: {len(pdf_files)}")
print(f"EasyOCR available: {easyocr_available}")
print("=" * 72)

records = []
for i, pdf_path in enumerate(pdf_files, start=1):
    txt, method, err_note = extract_text_from_pdf(
        pdf_path=pdf_path,
        max_pages=MAX_PAGES_PER_PDF,
        use_gpu=False,
        debug=False
    )

    char_count = len(txt)
    low_quality = char_count < LOW_QUALITY_THRESHOLD

    short_hash = hashlib.md5(str(pdf_path).encode("utf-8")).hexdigest()[:10]
    out_txt_name = f"{pdf_path.stem}_{short_hash}.txt"
    out_txt_path = OCR_TEXT_DIR / out_txt_name
    out_txt_path.write_text(txt, encoding="utf-8", errors="ignore")

    records.append({
        "file_path": str(pdf_path),
        "ocr_method": method,
        "char_count": char_count,
        "low_quality": low_quality,
        "text_file": str(out_txt_path),
        "error": err_note,
    })

    print(f"[{i:02d}/{len(pdf_files)}] method={method:18s} chars={char_count:5d} file={pdf_path.name}")

# -----------------------------
# 4) Save CSV summary
# -----------------------------
csv_path = OUT_DIR / "ocr_batch_results.csv"
fieldnames = ["file_path", "ocr_method", "char_count", "low_quality", "text_file", "error"]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    for r in records:
        w.writerow(r)

num_total = len(records)
num_pypdf = sum(1 for r in records if r["ocr_method"] == "pypdf")
num_ocr = sum(1 for r in records if r["ocr_method"] == "pdf2image+easyocr")
num_none = sum(1 for r in records if r["ocr_method"] == "none")
num_low = sum(1 for r in records if r["low_quality"])
num_nonempty = sum(1 for r in records if r["char_count"] > 0)

print("\n" + "=" * 72)
print("BATCH OCR COMPLETE")
print(f"Processed: {num_total}")
print(f"Non-empty text files: {num_nonempty}")
print(f"pypdf used: {num_pypdf}")
print(f"OCR used: {num_ocr}")
print(f"none: {num_none}")
print(f"Low-quality (<{LOW_QUALITY_THRESHOLD} chars): {num_low}")
print(f"CSV: {csv_path}")
print(f"Text dir: {OCR_TEXT_DIR}")
print("=" * 72)